# H2-1 확장 — 8단계: 위약검정 (캠페인 미수신 916가구)

**지금까지 확정된 것**: 9개 Type×계층 조합 중 **TypeC×고지출 하나만** BH 보정 후에도 유의
(coef=-6.25, p_adj=0.0033, 표본 295명 — 표본 문제는 없음).

**아직 남은 문제**: 이 결과가 "TypeC가 지출을 줄였다"인지, "회사가 이미 지출이 줄던 고지출
고객을 잡으려고 TypeC를 그 시점에 보낸 것"인지(역방향 타겟팅) 구분이 안 됨.
가구·주차 고정효과는 "이 가구 원래 수준"과 "이 주 공통 요인"은 걸러주지만
"왜 하필 이 시점에 이 가구가 캠페인을 받았는가"는 못 걸러냄.

**이번 단계**: 캠페인을 **한 번도 받지 않은** 916가구만 갖고, 같은 33~101주 관찰기간 동안
고지출 계층이 캠페인 없이도 저절로 지출이 줄어드는 패턴을 보이는지 확인.
만약 미수신 가구의 고지출 계층에서도 비슷한 감소가 나타난다면 → 방금 확정한 결과는
캠페인 효과가 아니라 자연적인 하락(회귀-평균 등)일 가능성이 높음.


## 0. 환경 설정

In [1]:
from importlib.metadata import version
print("linearmodels", version("linearmodels"), "| statsmodels", version("statsmodels"))

linearmodels 7.0 | statsmodels 0.14.6


In [2]:
import pandas as pd
import numpy as np
from linearmodels.panel import PanelOLS

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 140)

from config import (DATA_DIR, OUT_DIR, INTERMEDIATE_DIR, FIG_DIR, SEED, TIER_MIN_WEEK, TIER_MAX_WEEK,
                    CAMP_MIN_WEEK, CAMP_MAX_WEEK, MID_WEEK, CELL_SIZE_THRESH, TIER_EN,
                    load_tiers, load_panel, load_camp_full)



## 1~7단계 파이프라인 재구성 (동일 로직 — 새 내용 없음)

In [3]:
tier_df = load_tiers()
panel = load_panel()[["household_key", "WEEK_NO", "spend", "tier3"]]
recipients = set(tier_df.index[tier_df["will_receive"] == 1])
never_recipients = set(tier_df.index[tier_df["will_receive"] == 0])
print(f"캠페인 미수신 가구: {len(never_recipients):,}명")
print(f"전체 패널 shape: {panel.shape}")

캠페인 미수신 가구: 916명
전체 패널 shape: (172500, 4)


## 2. 미수신 가구만 추출 + 계층별 인원 확인

5단계에서 이미 확인했듯 고지출 계층의 미수신 가구는 49명뿐이라 표본이 작음 — 이 검정의
결과를 볼 때 이 점을 감안해야 함.

In [4]:
never_panel = panel[panel["household_key"].isin(never_recipients)].copy()

print(f"미수신 가구 패널 shape: {never_panel.shape}")
print()
print("[미수신 가구 계층별 인원수]")
tier_counts_never = never_panel.groupby("tier3", observed=True)["household_key"].nunique()
print(tier_counts_never)
print()
print("※ 고지출 계층 49명은 5단계와 동일한 표본 제약 -- 이 검정의 검정력도 약할 수 있음")


미수신 가구 패널 shape: (63204, 4)

[미수신 가구 계층별 인원수]
tier3
저지출    624
중지출    243
고지출     49
Name: household_key, dtype: int64

※ 고지출 계층 49명은 5단계와 동일한 표본 제약 -- 이 검정의 검정력도 약할 수 있음


## 3. 위약검정 A — 서술적 비교 (전반기 vs 후반기 평균 지출)

가장 단순한 확인: 캠페인 없이도 33~42주 대비 92~101주에 고지출 계층 지출이 줄어드는가.

In [5]:
early = never_panel[never_panel["WEEK_NO"].between(CAMP_MIN_WEEK, CAMP_MIN_WEEK + 9)]
late  = never_panel[never_panel["WEEK_NO"].between(CAMP_MAX_WEEK - 9, CAMP_MAX_WEEK)]

early_avg = early.groupby("tier3", observed=True)["spend"].mean()
late_avg  = late.groupby("tier3", observed=True)["spend"].mean()

placebo_desc = pd.DataFrame({
    "초반(33-42주) 평균": early_avg,
    "후반(92-101주) 평균": late_avg,
})
placebo_desc["변화액"] = placebo_desc["후반(92-101주) 평균"] - placebo_desc["초반(33-42주) 평균"]
placebo_desc["변화율(%)"] = (placebo_desc["변화액"] / placebo_desc["초반(33-42주) 평균"] * 100).round(1)
print("[미수신 가구 -- 계층별 전반기 대비 후반기 지출 변화 (서술적 비교)]")
print(placebo_desc.round(2))


[미수신 가구 -- 계층별 전반기 대비 후반기 지출 변화 (서술적 비교)]
       초반(33-42주) 평균  후반(92-101주) 평균   변화액  변화율(%)
tier3                                             
저지출             4.61           12.25  7.63   165.5
중지출            11.27           17.33  6.06    53.8
고지출            43.96           43.54 -0.42    -1.0


In [6]:
# --- source CSV: descriptive early vs late spending among never-recipients
_d = placebo_desc.rename(columns={"초반(33-42주) 평균": "mean_early", "후반(92-101주) 평균": "mean_late",
                                  "변화액": "change", "변화율(%)": "change_pct"}).rename(index=TIER_EN).rename_axis("tier").reset_index()
_d["n_households"] = _d["tier"].map(tier_counts_never.rename(index=TIER_EN))
_d["early_weeks"] = f"{CAMP_MIN_WEEK}-{CAMP_MIN_WEEK + 9}"
_d["late_weeks"] = f"{CAMP_MAX_WEEK - 9}-{CAMP_MAX_WEEK}"
_d.to_csv(OUT_DIR / "placebo_descriptive.csv", index=False)

## 4. 위약검정 B — 회귀 기반 (가구·주차 고정효과, 6단계와 동일한 방법론)

`spend ~ (중지출×후반기) + (고지출×후반기) + 가구고정효과 + 주차고정효과`, 클러스터 SE.

기준(저지출×후반기 및 전반기 전체)과 비교했을 때, **캠페인 없이도** 중지출·고지출 계층에서
후반기에 추가적인 증가/감소가 있는지 확인. 이 계수가 6단계의 TypeC×고지출 계수(-6.25)와
비슷한 크기·방향이면 경고 신호.

In [7]:
never_panel["late"] = (never_panel["WEEK_NO"] >= MID_WEEK).astype(int)
for t in ["중지출", "고지출"]:
    never_panel[f"{t}_x_late"] = ((never_panel["tier3"] == t) & (never_panel["late"] == 1)).astype(int)

placebo_cols = ["중지출_x_late", "고지출_x_late"]

reg_never = never_panel.copy()
reg_never["entity"] = reg_never["household_key"]
reg_never["time"] = reg_never["WEEK_NO"]
reg_never = reg_never.set_index(["entity", "time"])

placebo_model = PanelOLS(
    dependent=reg_never["spend"],
    exog=reg_never[placebo_cols],
    entity_effects=True,
    time_effects=True,
    drop_absorbed=True
)
placebo_result = placebo_model.fit(cov_type="clustered", cluster_entity=True)
print(placebo_result.summary)


                          PanelOLS Estimation Summary                           
Dep. Variable:                  spend   R-squared:                        0.0004
Estimator:                   PanelOLS   R-squared (Between):             -0.0218
No. Observations:               63204   R-squared (Within):              -0.0003
Date:                Mon, Oct 19 2026   R-squared (Overall):             -0.0089
Time:                        12:02:37   Log-likelihood                 -2.98e+05
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      12.216
Entities:                         916   P-value                           0.0000
Avg Obs:                       69.000   Distribution:                 F(2,62218)
Min Obs:                       69.000                                           
Max Obs:                       69.000   F-statistic (robust):             0.6398
                            

In [8]:
# --- source CSV: placebo regression (reference tier = low; week fixed effects absorb the common late-period change)
_pc = placebo_result.conf_int()
pd.DataFrame([{"term": term, "coef": placebo_result.params[term], "se": placebo_result.std_errors[term],
               "ci_low": _pc.loc[term, "lower"], "ci_high": _pc.loc[term, "upper"], "p_value": placebo_result.pvalues[term],
               "late_period_start_week": MID_WEEK, "n_obs": int(placebo_result.nobs),
               "n_households": int(placebo_result.entity_info["total"]),
               "n_households_in_tier": int(tier_counts_never.get("중지출" if term.startswith("중") else "고지출", 0))}
              for term in placebo_cols]).assign(term=lambda d: d["term"].map({"중지출_x_late": "mid_x_late", "고지출_x_late": "high_x_late"})) \
    .to_csv(OUT_DIR / "placebo_regression.csv", index=False)

## 5. 실제 효과와 위약 효과 나란히 비교

In [9]:
_fe = pd.read_csv(OUT_DIR / "fe_coefficients.csv")
_c = _fe[(_fe["type"] == "TypeC") & (_fe["tier"] == "high")].iloc[0]
_pcs = pd.read_csv(OUT_DIR / "placebo_regression.csv").set_index("term").loc["high_x_late"]
comparison = pd.DataFrame({
    "구분": ["[실제] TypeC x 고지출 (6단계)", "[위약] 고지출 x 후반기 (미수신, 8단계)"],
    "계수": [_c["coef"], placebo_result.params.get("고지출_x_late", np.nan)],
    "p값": [_c["p_raw"], placebo_result.pvalues.get("고지출_x_late", np.nan)],
    "표본": [int(_c["n_recipients"]), int(tier_counts_never.get("고지출", 0))],
})
pd.DataFrame([
    {"estimate": "actual_typeC_x_high", "coef": _c["coef"], "ci_low": _c["ci_low"], "ci_high": _c["ci_high"], "p_value": _c["p_raw"],
     "n_households": int(_c["n_recipients"])},
    {"estimate": "placebo_high_x_late", "coef": _pcs["coef"], "ci_low": _pcs["ci_low"], "ci_high": _pcs["ci_high"],
     "p_value": _pcs["p_value"], "n_households": int(_pcs["n_households_in_tier"])},
]).to_csv(OUT_DIR / "placebo_vs_actual.csv", index=False)
print(comparison.to_string(index=False))
print()
print("판단 기준:")
print("  위약 계수가 0에 가깝고 유의하지 않다  -> 실제 효과가 캠페인 때문일 가능성 유지")
print("  위약 계수도 비슷한 크기로 음수+유의  -> 실제 효과는 자연적 하락일 가능성 (역타겟팅 등)")


                       구분        계수       p값  표본
   [실제] TypeC x 고지출 (6단계) -6.252591 0.000371 295
[위약] 고지출 x 후반기 (미수신, 8단계) -4.740102 0.260913  49

판단 기준:
  위약 계수가 0에 가깝고 유의하지 않다  -> 실제 효과가 캠페인 때문일 가능성 유지
  위약 계수도 비슷한 크기로 음수+유의  -> 실제 효과는 자연적 하락일 가능성 (역타겟팅 등)


## 결과 해석 가이드

**확인할 순서**

1. 위약검정 B의 `고지출_x_late` 계수 부호와 유의성을 본다.
   - 유의하지 않거나 양수라면 → TypeC×고지출 결과가 캠페인 효과일 가능성이 유지됨
   - 음수이고 유의하다면 → 캠페인 없이도 고지출 계층은 원래 후반기에 지출이 주는 경향이 있다는 뜻,
     즉 6단계 결과가 캠페인 때문이 아니라 자연적 하락(혹은 타겟팅 시점 자체의 내생성)일 가능성이 커짐
2. **표본 49명이라는 제약을 항상 같이 언급**. 여기서 "유의하지 않다"가 나와도
   "위약효과가 없다는 강한 증거"가 아니라 "이 표본 크기로는 있어도 못 잡을 수 있다"는
   해석의 여지를 남겨야 함(5단계에서 이미 지적한 것과 동일한 한계).
3. 서술적 비교(3번)와 회귀 기반(4번) 결과의 방향이 같은지도 확인 — 다르면 후반기 정의(MID_WEEK)나
   구간 설정에 따라 결과가 민감하다는 뜻이므로 강건성 체크가 추가로 필요.

## 최종 결론 작성 시 표현

- 위약검정을 통과한 경우: "TypeC 캠페인 활성 주에 고지출 고객의 지출이 감소하는 현상이 관측되었으며,
  캠페인 미수신 가구에서는 같은 시기 유사한 패턴이 나타나지 않아(위약검정 통과) 이 결과가
  캠페인과 무관한 자연적 하락일 가능성은 낮아 보인다. 다만 표본 제약(고지출 계층 위약 비교군 49명)과
  캠페인 타겟팅 규칙 미상으로 인해 확정적 인과관계로 단정하지는 않는다."
- 위약검정에서 유사 패턴이 나타난 경우: "TypeC×고지출 효과는 캠페인 없이도 나타나는 자연적 하락과
  구분되지 않아, 인과적 해석의 근거로 사용하기 어렵다. 향후 무작위 파일럿으로 재검증이 필요하다."
